# RAG-IDEArq — Evaluación RAGAS con Langfuse

Evaluación del sistema RAG con:
- **Dataset v3**: 30 preguntas (15 simples + 15 complejas, multilingüe)
- **LLMs**: Phi-3.5-mini, Qwen3-4B-Instruct-2507, Llama-3.2-3B-Instruct
- **Embeddings**: all-MiniLM-L6-v2, gte-multilingual-base
- **Judge**: Mistral API (`mistral-small-latest`)
- **Monitor**: Langfuse (:4000)

## Sugerencias de mejora para los prompts (en Langfuse UI)

**`prompt_zero_shot`:**
- Añadir: "Si la pregunta menciona un año específico, prioriza docs publicados ±5 años"
- Añadir: "Si pide listas de yacimientos, devuelve: nombre, provincia, cronología"
- Añadir: "Cita siempre la fuente (autor, año) si aparece en contexto"
- Añadir: "Si no hay info, di: 'No hay información suficiente en el contexto para responder a esta pregunta.'"

**`prompt_one_shot`:**
- Cambiar ejemplo de isótopos Sr por uno de C14 + yacimientos (más representativo)
- Añadir: "Sigue el formato del ejemplo: viñetas cuando la pregunta pida listas"

**`prompt_few_shot`:**
- Añadir 3er ejemplo: pregunta C14 con cronología (anclar formato BP/calBC)
- Añadir: "Cuando menciones dataciones, incluye BP y calBC si están disponibles"

In [1]:
# Cell 1: Setup
import os
import sys
import re
import time
import json
import itertools
import logging
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any

# Add project root to path
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=True)

from src.config import (
    WEAVIATE_URL, EMBEDDINGS, collection_name,
    LLMS, LLM_TEMPERATURES, RESULTS_DIR,
    LANGFUSE_DATASET_NAME, PROMPT_NAMES,
    RERANKING_CONFIG,
)
from src.langfuse_monitor import (
    get_callback, get_prompt, get_or_create_dataset_v3, score_trace,
    langfuse_handler, langfuse_client, log_retrieval_breakdown,
)
from src.reranker import rerank_documents
from data.eval_questions import (
    get_eval_items, get_metadata, get_questions, get_ground_truths, get_sources,
)

import weaviate
from langchain_weaviate import WeaviateVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# RAGAS
from ragas import EvaluationDataset, evaluate
from ragas.metrics import (
    Faithfulness, ContextPrecision, ContextRecall, AnswerCorrectness,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.dataset_schema import SingleTurnSample
from ragas.run_config import RunConfig

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load embedding models
print("\nLoading embedding models...")
embedding_models = {}
for emb_key, emb_cfg in EMBEDDINGS.items():
    try:
        emb = HuggingFaceEmbeddings(
            model_name=emb_cfg["model_name"],
            model_kwargs={"device": "cuda"},
            encode_kwargs={"device": "cuda"},
        )
        embedding_models[emb_key] = emb
        print(f"  [OK] {emb_key} on CUDA")
    except Exception as e:
        if emb_cfg.get("trust_remote_code", False):
            try:
                emb = HuggingFaceEmbeddings(
                    model_name=emb_cfg["model_name"],
                    model_kwargs={"device": "cuda", "trust_remote_code": True},
                    encode_kwargs={"device": "cuda"},
                )
                embedding_models[emb_key] = emb
                print(f"  [OK] {emb_key} on CUDA (trust_remote_code)")
            except Exception as e2:
                emb = HuggingFaceEmbeddings(
                    model_name=emb_cfg["model_name"],
                    model_kwargs={"device": "cpu", "trust_remote_code": True},
                    encode_kwargs={"device": "cpu"},
                )
                embedding_models[emb_key] = emb
                print(f"  [CPU] {emb_key} (trust_remote_code)")
        else:
            emb = HuggingFaceEmbeddings(
                model_name=emb_cfg["model_name"],
                model_kwargs={"device": "cpu"},
                encode_kwargs={"device": "cpu"},
            )
            embedding_models[emb_key] = emb
            print(f"  [CPU] {emb_key}")

# Load LLM models
print("\nLoading LLM models...")
llm_models = {}
for llm_name, llm_model in LLMS.items():
    for temp in LLM_TEMPERATURES:
        key = f"{llm_name}_t{temp}"
        llm_models[key] = OllamaLLM(model=llm_model, temperature=temp)
        print(f"  [OK] {key}")

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Dataset: {LANGFUSE_DATASET_NAME}")
print(f"LLMs: {list(LLMS.keys())}")
print(f"Embeddings: {list(EMBEDDINGS.keys())}")
print(f"Temperatures: {LLM_TEMPERATURES}")
print(f"Reranking enabled: {RERANKING_CONFIG['enabled']}")
print(f"Rerank model: {RERANKING_CONFIG['model']}")
print(f"Langfuse handler: {langfuse_handler is not None}")


ImportError: cannot import name 'CallbackHandler' from 'langfuse' (/home/raglinux/env_rag/lib/python3.12/site-packages/langfuse/__init__.py)

In [2]:
# Cell 4: Build RAG chain with optional reranking
def build_rag_chain(emb_key, llm_name, prompt_key):
    """Build a RAG chain for one embedding/llm/prompt combination."""
    coll_name = collection_name(emb_key)
    if not w_client.collections.exists(coll_name):
        raise ValueError(f"Collection '{coll_name}' does not exist. Run indexing first.")

    vs = WeaviateVectorStore(
        client=w_client,
        index_name=coll_name,
        text_key="content",
        embedding=embedding_models[emb_key],
        attributes=["filename", "source", "chunk_index", "year", "language", "doi", "authors", "periodo", "region"],
    )

    k = RERANKING_CONFIG["k_retrieval"] if RERANKING_CONFIG["enabled"] else RERANKING_CONFIG["top_n"]
    retriever = vs.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )

    prompt_template = get_prompt(prompt_key)
    print(f"  Prompt '{prompt_key}' loaded from Langfuse")

    llm = llm_models[llm_name]

    return retriever, prompt_template, llm

def run_rag_with_rerank(retriever, prompt_template, llm, question, trace_id=None):
    """Run one RAG query with optional reranking and Langfuse tracing."""
    docs = retriever.invoke(question)

    # Log retrieval breakdown to Langfuse
    if trace_id:
        log_retrieval_breakdown(
            trace_id=trace_id,
            question=question,
            weaviate_docs=len(docs),
            metadata={"embedding": "MiniLM", "k": len(docs)},
        )

    if RERANKING_CONFIG["enabled"] and len(docs) > RERANKING_CONFIG["top_n"]:
        ranked = rerank_documents(
            question, docs,
            top_n=RERANKING_CONFIG["top_n"],
            reranker_model=RERANKING_CONFIG["model"]
        )
        selected_docs = [r["doc"] for r in ranked]
        print(f"    Reranked: {len(selected_docs)} docs from {len(docs)}")
    else:
        selected_docs = docs[:RERANKING_CONFIG["top_n"]]
        print(f"    No rerank: {len(selected_docs)} docs")

    context = "\n\n".join([d.page_content for d in selected_docs])
    if hasattr(prompt_template, 'format'):
        prompt_text = prompt_template.format(context=context, question=question)
    else:
        prompt_text = prompt_template.compile(context=context, question=question)

    # Generation with Langfuse handler
    answer = llm.invoke(prompt_text, config={"callbacks": [langfuse_handler]})

    return answer, selected_docs


In [3]:
# Cell 5: RAGAS evaluation setup
from langchain_mistralai import ChatMistralAI

mistral_judge = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0.1,
    max_retries=3,
    timeout=180,
)

ragas_llm = LangchainLLMWrapper(mistral_judge)
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)

ragas_metrics = [
    Faithfulness(llm=ragas_llm),
    ContextPrecision(llm=ragas_llm),
    ContextRecall(llm=ragas_llm),
    AnswerCorrectness(llm=ragas_llm, embeddings=ragas_embeddings),
]

ragas_run_config = RunConfig(
    max_workers=1,
    max_retries=10,
    max_wait=120,
    timeout=900,
)

print("RAGAS metrics configured:")
for m in ragas_metrics:
    print(f"  - {m.__class__.__name__}")


NameError: name 'LangchainLLMWrapper' is not defined

In [ ]:
# ── SMOKE TEST: Verificar Langfuse integration ──
print("="*60)
print("SMOKE TEST: Langfuse integration verification")
print("="*60)

SMOKE_EMBEDDING = "all-MiniLM-L6-v2"
SMOKE_LLM = "Phi-3.5-mini"
SMOKE_PROMPT = "zero_shot"
SMOKE_TEMP = 0.3
SMOKE_QUESTIONS = eval_items[:2]

print(f"Embedding: {SMOKE_EMBEDDING}")
print(f"LLM: {SMOKE_LLM}")
print(f"Prompt: {SMOKE_PROMPT}")
print(f"Temperatura: {SMOKE_TEMP}")
print(f"Preguntas: {len(SMOKE_QUESTIONS)}")
print(f"Rerank: {RERANKING_CONFIG['enabled']}")
print()

# 1. Test retrieval
print("[1/5] Test retrieval...")
retriever, prompt_template, llm = build_rag_chain(SMOKE_EMBEDDING, SMOKE_LLM, SMOKE_PROMPT)
llm_key = f"{SMOKE_LLM}_t{SMOKE_TEMP}"
llm = llm_models[llm_key]
docs = retriever.invoke(SMOKE_QUESTIONS[0]["input"]["question"])
print(f"  ✅ Retrieved {len(docs)} documents")
if docs:
    print(f"  First doc: {docs[0].metadata.get('filename', 'N/A')}")

# 2. Create Langfuse trace with input
print("\n[2/5] Creating Langfuse trace...")
question = SMOKE_QUESTIONS[0]["input"]["question"]
callback_info = get_callback(
    session_id="smoke_test",
    tags=["test", "smoke"],
    input_data={"question": question},
)
trace_id = callback_info.get("trace_id")
print(f"  ✅ Trace created: {trace_id}")

# 3. Test generation
print("\n[3/5] Test generation...")
context = "\n\n".join([d.page_content for d in docs[:3]])
if hasattr(prompt_template, 'format'):
    prompt_text = prompt_template.format(context=context, question=question)
else:
    prompt_text = prompt_template.compile(context=context, question=question)
test_answer = llm.invoke(prompt_text)
print(f"  ✅ Generated answer ({len(test_answer)} chars)")
print(f"  Preview: {test_answer[:100]}...")

# Update trace with output
update_trace_output(trace_id, {"answer": test_answer[:500]})
print(f"  ✅ Updated trace output")

# Log retrieval breakdown
log_retrieval_breakdown(
    trace_id=trace_id,
    question=question,
    weaviate_docs=len(docs),
    metadata={"embedding": "MiniLM", "k": len(docs)},
)
print(f"  ✅ Logged retrieval breakdown")

# 4. Test reranker
if RERANKING_CONFIG["enabled"]:
    print("\n[4/5] Test reranker...")
    ranked = rerank_documents(question, docs, top_n=RERANKING_CONFIG["top_n"], reranker_model=RERANKING_CONFIG["model"])
    print(f"  ✅ Reranked {len(ranked)} documents")
    print(f"  Top score: {ranked[0]['score']:.4f}")
else:
    print("\n[4/5] Test reranker... SKIPPED")

# 5. Test RAGAS and score in Langfuse
print("\n[5/5] Test RAGAS + Langfuse scoring...")
sample = SingleTurnSample(
    user_input=question,
    response=test_answer,
    reference=SMOKE_QUESTIONS[0]["expected_output"]["ground_truth"],
    retrieved_contexts=[d.page_content[:800] for d in docs[:5]],
)
eval_ds = EvaluationDataset(samples=[sample])

try:
    ragas_result = evaluate(dataset=eval_ds, metrics=ragas_metrics, llm=ragas_llm, embeddings=ragas_embeddings, run_config=ragas_run_config)
    print(f"  ✅ RAGAS scores:")
    for metric, value in ragas_result.scores[0].items():
        print(f"     {metric}: {value:.4f}")
        # Score in Langfuse
        try:
            score_trace(trace_id=trace_id, name=metric, value=value)
            print(f"     → Scored in Langfuse")
        except Exception as e:
            print(f"     ⚠️ Langfuse error: {e}")
except Exception as e:
    print(f"  ⚠️ RAGAS error: {e}")

# Verify Langfuse traces
print("\n[6/6] Verifying Langfuse traces...")
import requests
from requests.auth import HTTPBasicAuth
BASE_URL = os.getenv("LANGFUSE_BASE_URL", "http://localhost:4000")
PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
resp = requests.get(f"{BASE_URL}/api/public/traces?limit=3", auth=HTTPBasicAuth(PUBLIC_KEY, SECRET_KEY), timeout=10)
if resp.status_code == 200:
    traces = resp.json().get('data', [])
    print(f"  ✅ Found {len(traces)} traces in Langfuse")
    for t in traces[:2]:
        inp = t.get('input')
        out = t.get('output')
        scores = t.get('scores', [])
        print(f"\n    Session: {t.get('sessionId')}")
        print(f"    Input: {str(inp)[:100]}...")
        print(f"    Output: {str(out)[:100]}...")
        print(f"    Scores: {len(scores)}")
        for s in scores[:2]:
            print(f"      - {s.get('name')}: {s.get('value')}")
else:
    print(f"  ⚠️ Langfuse API error: {resp.status_code}")

print("\n" + "="*60)
print("✅ SMOKE TEST COMPLETADO")
print("="*60)
print("\nCheck Langfuse at http://localhost:4000")


In [ ]:
# Cell 6: Run full evaluation grid
eval_items = get_eval_items()
all_results = []

combos = list(itertools.product(
    EMBEDDINGS.keys(),
    LLMS.keys(),
    ["zero_shot", "one_shot", "few_shot"],
    LLM_TEMPERATURES,
))

print(f"\nTotal combos: {len(combos)}")
print(f"Questions per combo: {len(eval_items)}")
print(f"Total evaluations: {len(combos) * len(eval_items)}\n")

for emb_key, llm_name, prompt_key, temperature in combos:
    llm_key = f"{llm_name}_t{temperature}"
    combo_label = f"{emb_key}|{llm_name}|{prompt_key}|t{temperature}"
    print(f"\n{'='*60}")
    print(f"Combo: {combo_label}")
    print(f"{'='*60}")

    try:
        retriever, prompt_template, _ = build_rag_chain(emb_key, llm_name, prompt_key)
        llm = llm_models[llm_key]
    except ValueError as e:
        print(f"  SKIP: {e}")
        continue

    ragas_samples = []

    for idx, item in enumerate(eval_items):
        question = item["input"]["question"]
        ground_truth = item["expected_output"]["ground_truth"]
        source = item["expected_output"]["source"]
        tipo = item["input"]["tipo"]
        idioma = item["input"]["idioma"]
        n_articulos = item["input"]["n_articulos"]

        print(f"  Q{idx+1}/{len(eval_items)} [{tipo}/{idioma}]: {question[:60]}...")

        # Create Langfuse trace with input
        callback_info = get_callback(
            session_id=combo_label,
            tags=[emb_key, llm_name, prompt_key, f"t{temperature}", tipo, idioma],
            input_data={"question": question},
        )
        trace_id = callback_info.get("trace_id")

        t0 = time.time()
        try:
            answer, docs = run_rag_with_rerank(retriever, prompt_template, llm, question, trace_id)
            latency = time.time() - t0

            # Update trace with output
            if trace_id:
                update_trace_output(trace_id, {"answer": answer[:500]})

            contexts = [d.page_content[:800] for d in docs[:5]]
            sample = SingleTurnSample(
                user_input=question,
                response=answer,
                reference=ground_truth,
                retrieved_contexts=contexts,
            )
            ragas_samples.append(sample)

            all_results.append({
                "combo": combo_label,
                "embedding": emb_key,
                "llm": llm_name,
                "prompt": prompt_key,
                "temperature": temperature,
                "use_rerank": RERANKING_CONFIG["enabled"],
                "question": question,
                "answer": answer[:500] if answer else "",
                "ground_truth": ground_truth[:500],
                "source": source,
                "tipo": tipo,
                "idioma": idioma,
                "n_articulos": n_articulos,
                "n_docs_retrieved": len(docs),
                "latency_s": latency,
            })

            print(f"    OK ({latency:.1f}s, {len(docs)} docs)")

        except Exception as e:
            print(f"    ERROR: {e}")
            all_results.append({
                "combo": combo_label,
                "embedding": emb_key,
                "llm": llm_name,
                "prompt": prompt_key,
                "temperature": temperature,
                "use_rerank": RERANKING_CONFIG["enabled"],
                "question": question,
                "answer": f"Error: {e}",
                "ground_truth": ground_truth[:500],
                "source": source,
                "tipo": tipo,
                "idioma": idioma,
                "n_articulos": n_articulos,
                "n_docs_retrieved": 0,
                "latency_s": 0,
            })

    if ragas_samples:
        print(f"\n  Running RAGAS ({len(ragas_samples)} samples)...")
        eval_ds = EvaluationDataset(samples=ragas_samples)

        for attempt in range(5):
            try:
                ragas_result = evaluate(dataset=eval_ds, metrics=ragas_metrics, llm=ragas_llm, embeddings=ragas_embeddings, run_config=ragas_run_config)
                break
            except Exception as e:
                if "429" in str(e) or "rate_limit" in str(e).lower():
                    wait = 30 * (2 ** attempt)
                    print(f"  [Rate limit] Waiting {wait}s ({attempt+1}/5)...")
                    time.sleep(wait)
                else:
                    print(f"  [RAGAS Error] {e}")
                    ragas_result = None
                    break

        if ragas_result:
            print(f"  RAGAS results:")
            for metric_name, metric_value in ragas_result.scores[0].items():
                print(f"    {metric_name}: {metric_value:.4f}")

            # Score in Langfuse
            for metric_name, metric_value in ragas_result.scores[0].items():
                try:
                    score_trace(trace_id=trace_id, name=metric_name, value=metric_value)
                except Exception:
                    pass

    print(f"\n  Waiting 60s before next combo...")
    time.sleep(60)


In [ ]:
# Cell 7: Reporte segmentado
if not all_results:
    print("No results to report.")
else:
    df = pd.DataFrame(all_results)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = Path(RESULTS_DIR) / f"eval_v3_{timestamp}.csv"
    Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Results saved to: {output_path}")
    print(f"Total rows: {len(df)}")

    print("\n" + "="*60)
    print("Resultados por TIPO (simple vs compleja)")
    print("="*60)
    if 'faithfulness' in df.columns:
        print(df.groupby("tipo")[["faithfulness", "context_recall", "answer_correctness", "context_precision"]].mean())

    print("\n" + "="*60)
    print("Resultados por IDIOMA")
    print("="*60)
    if 'faithfulness' in df.columns:
        print(df.groupby("idioma")[["faithfulness", "context_recall", "answer_correctness", "context_precision"]].mean())

    print("\n" + "="*60)
    print("Answer Correctness por MODELO × TIPO")
    print("="*60)
    if 'answer_correctness' in df.columns:
        print(df.groupby(["llm", "tipo"])["answer_correctness"].mean().unstack())

    print("\n" + "="*60)
    print("Context Precision por EMBEDDING × TIPO")
    print("="*60)
    if 'context_precision' in df.columns:
        print(df.groupby(["embedding", "tipo"])["context_precision"].mean().unstack())

    print("\n" + "="*60)
    print("Latencia media por MODELO")
    print("="*60)
    print(df.groupby("llm")["latency_s"].mean().sort_values())

    print("\n" + "="*60)
    print("Resultados por TEMPERATURA")
    print("="*60)
    if 'faithfulness' in df.columns:
        print(df.groupby("temperature")[["faithfulness", "context_recall", "answer_correctness", "context_precision"]].mean())

    print("\n" + "="*60)
    print("RESULTADOS POR RERANKING")
    print("="*60)
    if "use_rerank" in df.columns and 'faithfulness' in df.columns:
        print(df.groupby("use_rerank")[["faithfulness", "context_recall", "answer_correctness", "context_precision"]].mean())

    print("\n" + "="*60)
    print("RESULTADOS POR RERANKING × IDIOMA")
    print("="*60)
    if "use_rerank" in df.columns and 'answer_correctness' in df.columns:
        print(df.groupby(["use_rerank", "idioma"])["answer_correctness"].mean().unstack())

    print("\n✅ Done! Check Langfuse UI at http://localhost:4000")
